In [1]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

SearchResult containing 18 data products.

 #     mission     year       author      exptime target_name distance
                                             s                 arcsec 
--- -------------- ---- ----------------- ------- ----------- --------
  0 TESS Sector 01 2018              SPOC     120   441420236      0.0
  1 TESS Sector 27 2020              SPOC      20   441420236      0.0
  2 TESS Sector 27 2020              SPOC     120   441420236      0.0
  3 TESS Sector 95 2025              SPOC      20   441420236      0.0
  4 TESS Sector 95 2025              SPOC     120   441420236      0.0
  5 TESS Sector 01 2018         TESS-SPOC    1800   441420236      0.0
  6 TESS Sector 27 2020         TESS-SPOC     600   441420236      0.0
  7 TESS Sector 01 2018               QLP    1800   441420236      0.0
  8 TESS Sector 27 2020               QLP     600   441420236      0.0
  9 TESS Sector 95 2025               QLP     200   441420236      0.0
 10 TESS Sector 27 2020           

In [8]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT
✓ 26 intervalo(s) carregado(s) de 'intervalos_flares_transitos_editavel.txt':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3886.130183, 3886.411659]
  6. [3888.081298, 3888.115210]
  7. [3888.784350, 3888.805991]
  8. [3889.179949, 3889.251286]
  9. [3889.826802, 3889.854535]
  10. [3891.013277, 3891.076262]
  11. [3891.741181, 3891.795356]
  12. [3891.844004, 3891.908792]
  13. [3891.950028, 3892.001562]
  14. [3892.631057, 3892.697367]
  15. [3892.812488, 3892.900441]
  16. [3893.017865, 3893.101673]
  17. [3893.403291, 3893.458281]
  18. [3896.687407, 3896.849926]
  19. [3897.567644, 3897.779646]
  20. [3899.153300, 3899.218797]
  21. [3899.542373, 3899.616046]
  22. [3900.664900, 3900.748400]
  23. [3900.906700, 3900.408000]
  24. [3904.512267, 3904.614076]
  25. [3904.782061, 3904.847219]
  26. [3905.340994, 3905.482584]

Máscara cri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PASSO 3: AJUSTE ROTACIONAL BASEADO EM CADÊNCIA + AJUSTES MANUAIS
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais e Lineares")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# FUNÇÕES PRINCIPAIS
# ============================================================

def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.5):
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    
    sigma_pontos = janela_pontos / 8
    
    gap_minimo_minutos = 20
    limite_gap = gap_minimo_minutos / (24 * 60)
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Filtro Base Global] Janela: {janela_horas}h | Sigma Clip: {sigma_clip_val}")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        
        for iteracao in range(10):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            residuo = f_seg - temp_smooth
            
            clipped = sigma_clip(
                residuo, 
                sigma_lower=10.0,
                sigma_upper=sigma_clip_val, 
                maxiters=1, 
                cenfunc='median', 
                stdfunc='mad_std'
            )            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
        
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, 
                              cadencia_s=120, janela_horas=5, 
                              sigma_upper=2.0, sigma_lower=3.0, iteracoes=6):
    """
    Usa a EXATA MESMA LÓGICA do modelo global, mas com parâmetros finos 
    aplicados apenas a um recorte, usando margem de segurança.
    """
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_calc = t[idx_calc]
    f_calc = f[idx_calc]
    mask_calc = mask_good_flares[idx_calc]
    
    pontos_por_hora = 3600 / cadencia_s
    sigma_pontos = (janela_horas * pontos_por_hora) / 8
    
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        residuo = f_calc - temp_smooth
        
        clipped = sigma_clip(residuo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, 
                             maxiters=1, cenfunc='median', stdfunc='mad_std')
        
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons = t_calc[mascara_combinada]
        f_bons = f_limpo[mascara_combinada]
        
        if len(t_bons) > 2:
            f_limpo = np.interp(t_calc, t_bons, f_bons)
        else:
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    
    inicio_corte = np.where(idx_calc == idx_alvo[0])[0][0]
    fim_corte = np.where(idx_calc == idx_alvo[-1])[0][0] + 1
    
    return idx_alvo, modelo_calc[inicio_corte:fim_corte]

# --- NOVA FUNÇÃO PARA FIT LINEAR ---
def ajustar_trecho_linear(t, f, mask_good_flares, t_inicio, t_fim, sigma_upper=2.0, sigma_lower=3.0, offset_y=0.0, tilt=0.0):
    """
    Traça uma reta no trecho e permite ajustes manuais de altura (offset) e inclinação (tilt).
    """
    from astropy.stats import sigma_clip
    
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_alvo = t[idx_alvo]
    f_alvo = f[idx_alvo]
    mask_alvo = mask_good_flares[idx_alvo]

    t_limpo = t_alvo[mask_alvo]
    f_limpo = f_alvo[mask_alvo]

    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    if np.sum(bons) > 2:
        coefs = np.polyfit(t_limpo[bons], f_limpo[bons], deg=1)
        
        # --- CONTROLE MANUAL DA RETA ---
        inclinacao_original = coefs[0]
        nova_inclinacao = inclinacao_original + tilt
        
        # Ponto de giro (centro do trecho no eixo x)
        t_centro = np.mean(t_alvo)
        y_centro = np.polyval(coefs, t_centro)
        
        # Equação da reta rotacionada e transladada: y = m*(x - x0) + y0 + offset
        modelo_linear = nova_inclinacao * (t_alvo - t_centro) + y_centro + offset_y
        
    else:
        modelo_linear = (np.ones_like(t_alvo) * np.nanmedian(f_alvo)) + offset_y

    return idx_alvo, modelo_linear
# -----------------------------------
# -----------------------------------

def costurar_bordas(t, modelo, bordas, tamanho_janela=25, sigma_gauss=0):
    from scipy.ndimage import gaussian_filter1d
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    
    for borda_idx in sorted(bordas):
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        # Reduzindo a restrição de tamanho mínimo já que temos menos pontos
        if (fim - inicio) < 10: continue
            
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 2, borda_idx - meia_zona)
        ini_dir = min(fim - 2, borda_idx + meia_zona)
        
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 5 or len(idx_dir) < 5: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [n]):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado

def costurar_com_gaps(t, modelo, bordas, limite_gap_fator=5, tamanho_janela=5, sigma_gauss=3):
    from scipy.ndimage import gaussian_filter1d
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * limite_gap_fator
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]

    modelo_costurado = costurar_bordas(
        t, modelo, bordas,
        tamanho_janela=tamanho_janela,
        sigma_gauss=0 
    )

    if sigma_gauss > 0:
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)

    return modelo_costurado

# ============================================================
# GERANDO A BASE OFICIAL GLOBAL
# ============================================================

# Modelo global atualizado para 120s
modelo_manchas = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=120, 
    janela_horas=10, 
    sigma_clip_val=1.8
)

bordas_indices = set()

# ============================================================
# REGIÕES DE AJUSTE FINO E LINEAR
# ============================================================

# 1. Ajustes locais via Gaussiana [t_inicio, t_fim, janela_horas, sigma_upper, sigma_lower, iteracoes]
regioes_ajuste_local = [
    [3884.6271, 3884.9503, 10.0, 1.2, 3.0, 12],
    [3885.9927, 3886.4632, 10.0, 5, 5.0, 12],
    #[3900.1457, 3901.0141, 10.0, 1.5, 1.0, 10],
    [3900.1457, 3901.0141, 10.0, 0.8, 0.8, 10],
    [3902.9503, 3903.4152, 10.0, 0.8, 0.8, 10]

]

# 2. Ajustes locais via Reta Linear [t_inicio, t_fim, sigma_upper, sigma_lower]
regioes_ajuste_linear = [
   [3906.9180,3907.1300, 15.5, 9.0, 0.0001, 0.0005]
]

# Aplicando os ajustes Gaussianos
for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    idx_alvo, mod_local = ajustar_trecho_especifico(
        t, f, mask_good_flares, t_inicio=ini, t_fim=fim, 
        cadencia_s=120, # Atualizado aqui também
        janela_horas=janela_h, sigma_upper=sig_up, sigma_lower=sig_low, iteracoes=iters
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_local
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# Aplicando os ajustes Lineares
for ini, fim, sig_up, sig_low, off_y, tilt_val in regioes_ajuste_linear:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    idx_alvo, mod_linear = ajustar_trecho_linear(
        t, f, mask_good_flares, t_inicio=ini, t_fim=fim, 
        sigma_upper=sig_up, sigma_lower=sig_low, 
        offset_y=off_y, tilt=tilt_val 
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_linear
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# ============================================================
# COSTURA FINAL DAS REGIÕES LOCAIS E LINEARES
# ============================================================

if len(bordas_indices) > 0:
    print(f"\nCosturando {len(bordas_indices)} bordas de ajustes locais e lineares...")
    modelo_manchas_suave = costurar_bordas(
        t, modelo_manchas, sorted(list(bordas_indices)),
        tamanho_janela=25, # Reduzido (era 150), proporcional à nova cadência
        sigma_gauss=3      # Reduzido (era 20), proporcional à nova cadência
    )
else:
    print("\nNenhum ajuste local aplicado. Usando o modelo global liso.")
    from scipy.ndimage import gaussian_filter1d
    modelo_manchas_suave = np.copy(modelo_manchas)
    
    dt_mediano = np.median(np.diff(t))
    quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
    for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
        if i1 - i0 > 1:
            modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=3) # Atualizado aqui também


# ============================================================
# RESÍDUO E GRÁFICOS
# ============================================================

%matplotlib qt

import matplotlib.pyplot as plt

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, color='red', lw=2.2, label='Modelo Final')

# Pinta de azul as regiões do ajuste local (gaussiano)
for ini, fim, *_ in regioes_ajuste_local:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.15, label='Ajuste Local Gaussiano' if ini == regioes_ajuste_local[0][0] else "")

# Pinta de laranja a região do ajuste linear
if len(regioes_ajuste_linear) > 0:
    for ini, fim, *_ in regioes_ajuste_linear:
        ax1.axvspan(ini, fim, color='orange', alpha=0.25, label='Ajuste Linear')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_ajuste_local:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.15)
for ini, fim, *_ in regioes_ajuste_linear:
    ax2.axvspan(ini, fim, color='orange', alpha=0.25)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# EXPORTA TEMPO + RESÍDUO PARA .TXT
# ============================================================
arquivo_saida = "residuo_setor95.txt"
np.savetxt(
    arquivo_saida,
    np.column_stack([t, residual_manchas]),
    header="tempo_BTJD  residuo_normalizado",
    fmt="%.8f  %.8f"
)
print(f"\nArquivo salvo: {arquivo_saida}  ({len(t)} pontos)")

arquivo_saida = "lc_setor95.txt"
np.savetxt(
    arquivo_saida,
    np.column_stack([t, f]),
    header="tempo_BTJD  fluxo",
    fmt="%.8f  %.8f"
)
print(f"\nArquivo salvo: {arquivo_saida}  ({len(t)} pontos)")





AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais e Lineares

[Filtro Base Global] Janela: 10h | Sigma Clip: 1.8

Costurando 10 bordas de ajustes locais e lineares...

OK — Ajuste concluído.

Arquivo salvo: residuo_setor95.txt  (15111 pontos)

Arquivo salvo: lc_setor95.txt  (15111 pontos)


: 

In [7]:
# ============================================================
# ESTE COMANDO FAZ O MATPLOTLIB ABRIR NUMA JANELA SEPARADA
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# GRÁFICO DOBRADO EM FASE (DADOS NORMAIS E MODELO FINO)
# ============================================================

periodo = 4.86  # Período da AU Mic

# --- 1. CÁLCULO DA FASE PARA TODOS OS DADOS (NORMAL, COM FLARES) ---
fase = ((t - t[0]) % periodo) / periodo

# --- 2. PREPARANDO O MODELO PARA O GRÁFICO DE FASE (A LINHA FINA) ---
# Pegamos apenas o PRIMEIRO ciclo de rotação (1 período)
mask_um_ciclo = t < (t[0] + periodo)
fase_um_ciclo = ((t[mask_um_ciclo] - t[0]) % periodo) / periodo
modelo_um_ciclo = modelo_manchas_suave[mask_um_ciclo]

# Ordenamos esse único ciclo para a linha não cruzar
indices_ordenados = np.argsort(fase_um_ciclo)
fase_modelo_fina = fase_um_ciclo[indices_ordenados]
modelo_fino = modelo_um_ciclo[indices_ordenados]

# --- 3. MONTANDO O GRÁFICO ÚNICO ---
fig, ax = plt.subplots(figsize=(14, 6))

# Plotando os dados normais em fase
ax.plot(fase, f, 'k.', ms=2.0, alpha=0.5, label='Dados Originais em Fase')

# Plotando o modelo (1 ciclo) por cima dos dados
ax.plot(fase_modelo_fina, modelo_fino, color='red', lw=2.0, alpha=1.0, zorder=10, label='Modelo em Fase')

# ============================================================
# DEFININDO OS LIMITES DOS EIXOS (X e Y)
# ============================================================
# Fase vai estritamente de 0 a 1
ax.set_xlim(0.0, 1.0)

# Ajuste os valores abaixo para controlar o limite vertical (Fluxo)
# Baseado nas suas imagens, 0.94 a 1.06 costuma enquadrar bem a curva da AU Mic
ax.set_ylim(0.90, 1.10) 

ax.set_ylabel("Fluxo")
ax.set_xlabel(f"Fase (0.0 a 1.0) | Período = {periodo} dias")
ax.set_title("Curva Dobrada em Fase (AU Mic - Dados Normais e Modelo)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [9]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# PARAMETROS DOS PLANETAS (Plavchan et al. 2020)
# ============================================================

# AU Mic b
T0_b    = 1330.39051   # TBJD (BJD - 2457000)
P_b     = 8.463000     # dias
dur_b   = 3.50 / 24.0  # duracao em dias

# AU Mic c
T0_c    = 1342.2223    # TBJD
P_c     = 18.859019    # dias
dur_c   = 4.5  / 24.0  # duracao em dias

# ============================================================
# FUNCAO: centros de transito no intervalo de t
# ============================================================

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

t_min, t_max = t.min(), t.max()
centers_b = transit_times(T0_b, P_b, t_min, t_max)
centers_c = transit_times(T0_c, P_c, t_min, t_max)

print(f'Transitos AU Mic b: {len(centers_b)}')
for tc in centers_b: print(f'  BTJD = {tc:.5f}')
print(f'Transitos AU Mic c: {len(centers_c)}')
for tc in centers_c: print(f'  BTJD = {tc:.5f}')

# ============================================================
# FIGURA: 2 paineis
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ---- Painel superior: dados + modelo ----
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2, label='Modelo (manchas)', zorder=3)

# Mascara de flares/transitos (CSV)
_lm = False
for ini, fim in mascara_flares_list:
    lbl = 'Flares e Transitos detectados (IV)' if not _lm else '_nolegend_'
    ax1.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm = True

# Transitos AU Mic b
_lb = False
for tc in centers_b:
    lbl = 'Transito AU Mic b' if not _lb else '_nolegend_'
    ax1.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax1.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb = True

# Transitos AU Mic c
_lc = False
for tc in centers_c:
    lbl = 'Transito AU Mic c' if not _lc else '_nolegend_'
    ax1.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax1.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc = True

ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('AU Mic - Ajuste de Manchas, Mascara e Transitos Planetarios', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.25)

# ---- Painel inferior: residual ----
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.6)

_lm2 = False
for ini, fim in mascara_flares_list:
    lbl = 'Mascara' if not _lm2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm2 = True

_lb2 = False
for tc in centers_b:
    lbl = 'AU Mic b' if not _lb2 else '_nolegend_'
    ax2.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax2.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb2 = True

_lc2 = False
for tc in centers_c:
    lbl = 'AU Mic c' if not _lc2 else '_nolegend_'
    ax2.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax2.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc2 = True

ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Transitos AU Mic b: 3
  BTJD = 3886.21651
  BTJD = 3894.67951
  BTJD = 3903.14251
Transitos AU Mic c: 2
  BTJD = 3888.18986
  BTJD = 3907.04888


In [27]:
import numpy as np
import matplotlib.pyplot as plt
import batman

# ============================================================
# 1. PARÂMETROS DO ARTIGO COM BARRAS DE ERRO (AU Mic b)
# ============================================================
T0_b     = 3886.2646   
T0_b_err = 0.00015       # Incerteza do T0 da sua tabela
P_b      = 8.463000     
P_b_err  = 0.000002      # Incerteza do Período da sua tabela
dur_b    = 3.50 / 24.0   

# Parâmetros físicos para o Batman
params_b = batman.TransitParams()
params_b.t0 = T0_b; params_b.per = P_b; params_b.rp = 0.0496
params_b.a = 19.1; params_b.inc = 89.5; params_b.ecc = 0.0; params_b.w = 90.0
params_b.u = [0.32, 0.18]; params_b.limb_dark = "quadratic"

m_b = batman.TransitModel(params_b, t)
fluxo_teorico_b = m_b.light_curve(params_b)

# ============================================================
# 2. FUNÇÃO: CALCULAR CENTROS, PROPAGAR ERROS E FILTRAR DADOS
# ============================================================
def mapear_transitos_validos(T0, T0_err, P, P_err, t_array, janela_dias):
    t_min, t_max = t_array.min(), t_array.max()
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    
    eventos_validos = []
    
    for n in range(n_min, n_max + 1):
        tc = T0 + n * P
        # Fórmula de propagação de erro sigma_tc = sqrt( sigma_t0^2 + (n * sigma_p)^2 )
        tc_err = np.sqrt(T0_err**2 + (n * P_err)**2)
        
        # Verifica se existem pontos de dados nesta janela específica
        janela_indices = (t_array >= tc - janela_dias) & (t_array <= tc + janela_dias)
        pontos_na_janela = np.sum(janela_indices)
        
        # Só adiciona o trânsito se houver dados reais nele (limiar de segurança: > 5 pontos)
        if pontos_na_janela > 5:
            eventos_validos.append({'epoch': n, 'tc': tc, 'tc_err': tc_err})
        else:
            print(f"⚠️ Evento omitido (sem dados na lacuna): Época n={n} ao redor de BTJD={tc:.2f}")
            
    return eventos_validos

# Mapeia apenas onde o TESS realmente coletou dados para o planeta b
transitos_validos_b = mapear_transitos_validos(T0_b, T0_b_err, P_b, P_b_err, t, 0.25)

# ============================================================
# 3. PLOTAGEM DOS GRÁFICOS INDIVIDUAIS FILTRADOS
# ============================================================
%matplotlib qt

n_plots = len(transitos_validos_b)

if n_plots == 0:
    print("Nenhum trânsito com dados foi encontrado no intervalo.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5), sharey=True)
    if n_plots == 1: axes = [axes] # Garante comportamento de lista se for apenas 1 plot

    for i, ev in enumerate(transitos_validos_b):
        ax = axes[i]
        tc = ev['tc']
        tc_err = ev['tc_err']
        
        # Recorta a janela visual
        janela = (t >= tc - 0.25) & (t <= tc + 0.25)
        
        # Plota os dados e o modelo
        ax.plot(t[janela], residual_manchas[janela], 'k.', ms=3, alpha=0.4, label='Resíduos')
        ax.plot(t[janela], fluxo_teorico_b[janela], color='royalblue', lw=2.5, label='Modelo b')
        
        # Linhas de referência geométrica
        ax.axvline(tc, color='royalblue', ls='--', alpha=0.5)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
        
        # ─── BARRA DE ERRO DO CENTRO DO TRÂNSITO ───
        # Coloca um ponto com barra de erro horizontal (xerr) no topo do gráfico (y=1.002)
        ax.errorbar(tc, 1.002, xerr=tc_err, fmt='o', color='red', elinewidth=2, capsize=3, 
                    ms=4, label='Incerteza $T_c$ ($\sigma$)')
        
        ax.set_title(f"Trânsito Real (Época {ev['epoch']})\nCentro: {tc:.4f} d", fontsize=10, fontweight='bold')
        ax.set_xlabel('Tempo [BTJD]', fontsize=10)
        ax.grid(alpha=0.15)
        
        if i == 0:
            ax.set_ylabel('Fluxo Residual', fontsize=12)
            ax.legend(loc='lower left', fontsize=9)

    plt.suptitle('AU Mic b — Trânsitos Individuais Existentes (Incerteza do Centro em Vermelho)', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

⚠️ Evento omitido (sem dados na lacuna): Época n=1 ao redor de BTJD=3894.73


In [28]:
import numpy as np
import matplotlib.pyplot as plt
import batman

# ============================================================
# 1. CONFIGURAÇÃO DO MODELO BATMAN PARA O PLANETA c
# ============================================================
params_c = batman.TransitParams()
params_c.t0 = T0_c             # 1342.2223
params_c.per = P_c             # 18.859019
params_c.rp = 0.0395           # Rp/R* (Mais raso que o b)
params_c.a = 29.0              # a/R* (Mais afastado que o b)
params_c.inc = 89.0            # Inclinação
params_c.ecc = 0.0
params_c.w = 90.0
params_c.u = [0.32, 0.18]
params_c.limb_dark = "quadratic"

# Inicializa e calcula o modelo para todo o vetor de tempo
m_c = batman.TransitModel(params_c, t)
fluxo_teorico_c = m_c.light_curve(params_c)

# ============================================================
# 2. PLOTAGEM DOS TRÂNSITOS INDIVIDUAIS DE AU Mic c
# ============================================================
%matplotlib qt

n_transitos_c = len(centers_c)

fig, axes = plt.subplots(1, n_transitos_c, figsize=(5 * n_transitos_c, 4.5), sharey=True)

if n_transitos_c == 1:
    axes = [axes]

for i, tc in enumerate(centers_c):
    ax = axes[i]
    
    # Recorta uma janela de aproximadamente 14 horas (+- 0.30 dias) ao redor do centro
    janela = (t >= tc - 0.30) & (t <= tc + 0.30)
    
    # Plota os dados reais e o modelo teórico do planeta c
    ax.plot(t[janela], residual_manchas[janela], 'k.', ms=3, alpha=0.4, label='Resíduos')
    ax.plot(t[janela], fluxo_teorico_c[janela], color='seagreen', lw=2.5, label='Modelo c')
    
    ax.axvline(tc, color='seagreen', ls='--', alpha=0.6)
    ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
    
    ax.set_title(f'Evento {i+1}\nCentro: {tc:.4f} dias', fontsize=11, fontweight='bold')
    ax.set_xlabel('Tempo [BTJD]', fontsize=10)
    ax.grid(alpha=0.2)
    
    if i == 0:
        ax.set_ylabel('Fluxo Residual', fontsize=12)
        ax.legend(loc='lower left', fontsize=9)

plt.suptitle('AU Mic c — Inspeção de Trânsitos Individuais', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import batman
from scipy.optimize import curve_fit

# ============================================================
# NOTA: Certifique-se de que seus dados reais ('t' e 'residual_manchas')
# já estão carregados no seu notebook/script antes de rodar este bloco.
# ============================================================

# ============================================================
# 1. PARÂMETROS DO ARTIGO (Valores exatos obtidos via MCMC para AU Mic b)
# ============================================================
T0_b     = 1330.39051   # Tempo de conjunção central em TBJD
T0_b_err = 0.00015       # Incerteza do T0 vinda da Tabela 2
P_b      = 8.463000     # Período orbital em dias
P_b_err  = 0.000002      # Incerteza do Período vinda da Tabela 2

# Demais parâmetros fixos do planeta b conforme a Tabela 2 do artigo
RP_RSTAR_b = 0.0526
A_RSTAR_b  = 19.1
INC_b      = 89.5
ECC_b      = 0.0
W_b        = 90.0
LDC_b      = [0.13, 0.58] # Coeficientes linear (u0) e quadrático (u1)

# ============================================================
# 2. FUNÇÃO MODELO PARA O CURVE_FIT (Apenas o Centro T0 é Livre)
# ============================================================
def modelo_ajuste_t0(t_janela, t0_livre):
    """
    Função auxiliar que o curve_fit usará para testar valores de t0.
    Mantém os outros parâmetros físicos do planeta fixados conforme o artigo.
    """
    params_b = batman.TransitParams()
    params_b.t0 = t0_livre               # Este é o único parâmetro que o algoritmo vai mudar
    params_b.per = P_b
    params_b.rp = RP_RSTAR_b
    params_b.a = A_RSTAR_b
    params_b.inc = INC_b
    params_b.ecc = ECC_b
    params_b.w = W_b
    params_b.u = LDC_b
    params_b.limb_dark = "quadratic"
    
    m_b = batman.TransitModel(params_b, t_janela)
    return m_b.light_curve(params_b)

# ============================================================
# 3. FUNÇÃO: CALCULAR CENTROS TEÓRICOS E FILTRAR TRÂNSITOS COM DADOS
# ============================================================
def mapear_transitos_validos(T0, T0_err, P, P_err, t_array, janela_dias):
    t_min, t_max = t_array.min(), t_array.max()
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    
    eventos_validos = []
    
    for n in range(n_min, n_max + 1):
        tc = T0 + n * P
        # Fórmula de propagação de erro teórica (acumula com as épocas n)
        tc_err = np.sqrt(T0_err**2 + (n * P_err)**2)
        
        # Verifica se existem pontos de dados nesta janela específica
        janela_indices = (t_array >= tc - janela_dias) & (t_array <= tc + janela_dias)
        pontos_na_janela = np.sum(janela_indices)
        
        # Só adiciona o trânsito se houver dados reais nele (limiar de segurança: > 5 pontos)
        if pontos_na_janela > 5:
            eventos_validos.append({'epoch': n, 'tc': tc, 'tc_err': tc_err})
        else:
            print(f"⚠️ Evento omitido (sem dados na lacuna): Época n={n} ao redor de BTJD={tc:.2f}")
            
    return eventos_validos

# Mapeia apenas onde o TESS realmente coletou dados para o planeta b usando uma janela de 0.25 dias
transitos_validos_b = mapear_transitos_validos(T0_b, T0_b_err, P_b, P_b_err, t, 0.25)

# ============================================================
# 4. AJUSTE NÃO-LINEAR E PLOTAGEM DOS GRÁFICOS INDIVIDUAIS
# ============================================================
%matplotlib qt

n_plots = len(transitos_validos_b)

if n_plots == 0:
    print("Nenhum trânsito com dados foi encontrado no intervalo.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5), sharey=True)
    if n_plots == 1: axes = [axes] # Garante comportamento de lista se for apenas 1 plot

    for i, ev in enumerate(transitos_validos_b):
        ax = axes[i]
        tc_teorico = ev['tc']
        
        # Recorta estritamente os dados da janela visual atual (~6 horas para cada lado)
        janela = (t >= tc_teorico - 0.25) & (t <= tc_teorico + 0.25)
        t_f = t[janela]
        fluxo_f = residual_manchas[janela]
        
        # ─── MINIMIZAÇÃO DE MÍNIMOS QUADRADOS (AJUSTE DO CENTRO) ───
        try:
            # curve_fit ajustará o t0_livre baseado nos resíduos fotométricos reais
            # p0: chute inicial baseado na predição teórica recalculada
            # bounds: limita o algoritmo a procurar num raio seguro de ~2 horas (0.08 dias)
            popt, pcov = curve_fit(
                modelo_ajuste_t0, 
                t_f, 
                fluxo_f, 
                p0=[tc_teorico], 
                bounds=(tc_teorico - 0.08, tc_teorico + 0.08)
            )
            
            tc_medido = popt[0]
            # A variância do parâmetro é extraída da diagonal da matriz de covariância (pcov)
            tc_err_medido = np.sqrt(pcov[0,0])
            
            print(f"Época {ev['epoch']}:")
            print(f"  T_c Teórico (Predição): {tc_teorico:.5f} d")
            print(f"  T_c Medido  (Ajustado): {tc_medido:.5f} ± {tc_err_medido:.5f} d\n")
            
        except Exception as e:
            print(f"⚠️ Não foi possível ajustar o trânsito da Época {ev['epoch']}: {e}")
            # Se o ajuste falhar por falta de convergência, mantém o teórico original como fallback
            tc_medido = tc_teorico
            tc_err_medido = ev['tc_err']
        
        # Gera a curva teórica otimizada com base no centro REAL que foi medido
        fluxo_modelo_ajustado = modelo_ajuste_t0(t_f, tc_medido)
        
        # Plota os dados e o novo modelo ajustado
        ax.plot(t_f, fluxo_f, 'k-.', ms=3, alpha=0.4, label='Resíduos')
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5, label='Modelo Ajustado')
        
        # Linhas de referência geométrica baseadas no centro real medido
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
        
        # ─── BARRA DE ERRO COM BASE NOS DADOS REAIS ───
        # Coloca o ponto vermelho no topo com a incerteza estatística gerada pelo curve_fit
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', elinewidth=2, capsize=3, 
                    ms=4, label='Incerteza $T_c$ Real ($\sigma$)')
        
        ax.set_title(f"Trânsito Real (Época {ev['epoch']})\nCentro Medido: {tc_medido:.4f} d", fontsize=10, fontweight='bold')
        ax.set_xlabel('Tempo [BTJD]', fontsize=10)
        ax.grid(alpha=0.15)
        
        if i == 0:
            ax.set_ylabel('Fluxo Residual', fontsize=12)
            ax.legend(loc='lower left', fontsize=9)

    plt.suptitle('AU Mic b — Determinação de Centros de Trânsito via Ajuste Fotométrico (Valores de Martioli+2021)', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

⚠️ Evento omitido (sem dados na lacuna): Época n=303 ao redor de BTJD=3894.68
Época 302:
  T_c Teórico (Predição): 3886.21651 d
  T_c Medido  (Ajustado): 3886.26584 ± 0.00051 d

Época 304:
  T_c Teórico (Predição): 3903.14251 d
  T_c Medido  (Ajustado): 3903.18781 ± 0.00164 d



In [34]:
# ============================================================
# 4. AJUSTE NÃO-LINEAR E PLOTAGEM DOS GRÁFICOS (APENAS TRÂNSITOS)
# ============================================================
%matplotlib qt

n_plots = len(transitos_validos_b)

if n_plots == 0:
    print("Nenhum trânsito com dados foi encontrado no intervalo.")
else:
    # Layout alterado: n_plots linhas (trânsitos empilhados verticalmente) e apenas 1 coluna
    fig, axes = plt.subplots(n_plots, 1, figsize=(6.5, 4 * n_plots), squeeze=False)

    for i, ev in enumerate(transitos_validos_b):
        ax = axes[i, 0]   # Seleciona o eixo da linha atual
        tc_teorico = ev['tc']
        
        # Recorta os dados da janela atual
        janela = (t >= tc_teorico - 0.25) & (t <= tc_teorico + 0.25)
        t_f = t[janela]
        fluxo_f = residual_manchas[janela]
        
        # ─── MINIMIZAÇÃO DE MÍNIMOS QUADRADOS (AJUSTE DO CENTRO) ───
        try:
            popt, pcov = curve_fit(
                modelo_ajuste_t0, 
                t_f, 
                fluxo_f, 
                p0=[tc_teorico], 
                bounds=(tc_teorico - 0.08, tc_teorico + 0.08)
            )
            
            tc_medido = popt[0]
            tc_err_medido = np.sqrt(pcov[0,0])
            
            print(f"Época {ev['epoch']}:")
            print(f"  T_c Teórico (Predição): {tc_teorico:.5f} d")
            print(f"  T_c Medido  (Ajustado): {tc_medido:.5f} ± {tc_err_medido:.5f} d\n")
            
        except Exception as e:
            print(f"⚠️ Não foi possível ajustar o trânsito da Época {ev['epoch']}: {e}")
            tc_medido = tc_teorico
            tc_err_medido = ev['tc_err']
        
        # Gera a curva teórica otimizada com base no centro real medido
        fluxo_modelo_ajustado = modelo_ajuste_t0(t_f, tc_medido)
        
        # ─── DELIMITAÇÃO AUTOMÁTICA DO INÍCIO E FIM DO TRÂNSITO ───
        em_transito = fluxo_modelo_ajustado < 0.99999
        if np.any(em_transito):
            t_inicio = t_f[em_transito].min()
            t_fim = t_f[em_transito].max()
            
            # Linhas verticais azuis pontilhadas marcando o ingresso e egresso
            ax.axvline(t_inicio, color='royalblue', ls=':', lw=1.5, label='Início/Fim Trânsito' if i==0 else "")
            ax.axvline(t_fim, color='royalblue', ls=':', lw=1.5)
        
        # ─── PLOTAGEM DA CURVA DE LUZ ───
        # Mantido o padrão 'k.-' com transparência suave para não poluir visualmente as linhas
        ax.plot(t_f, fluxo_f, 'k.-', ms=4, lw=0.6, alpha=0.4, label='Dados (Sem Manchas)')
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5, label='Modelo Ajustado')
        
        # Linhas de referência geométrica (Centro medido e Linha de base 1.0)
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
        
        # Barra de erro do Tc ajustado no topo do trânsito
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', elinewidth=2, capsize=3, 
                    ms=4, label='Incerteza $T_c$ Real ($\sigma$)')
        
        # Customização de rótulos e eixos
        ax.set_title(f"Trânsito Real (Época {ev['epoch']}) — Centro Medido: {tc_medido:.4f} d", fontsize=10, fontweight='bold')
        ax.set_xlabel('Tempo [BTJD]', fontsize=9)
        ax.set_ylabel('Fluxo Residual', fontsize=10)
        ax.grid(alpha=0.15)
        
        # Exibe a legenda apenas no primeiro gráfico para poupar espaço
        if i == 0: 
            ax.legend(loc='lower left', fontsize=8)

    plt.suptitle('AU Mic b — Determinação de Centros de Trânsito via Ajuste Fotométrico (Valores de Martioli+2021)', fontsize=11, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

Época 302:
  T_c Teórico (Predição): 3886.21651 d
  T_c Medido  (Ajustado): 3886.26584 ± 0.00051 d

Época 304:
  T_c Teórico (Predição): 3903.14251 d
  T_c Medido  (Ajustado): 3903.18781 ± 0.00164 d



In [35]:
import numpy as np
import matplotlib.pyplot as plt
import batman
from scipy.optimize import curve_fit

# ============================================================
# NOTA: Certifique-se de que seus dados reais ('t' e 'residual_manchas')
# já estão carregados no seu ambiente antes de rodar este bloco.
# ============================================================

# ============================================================
# 1. PARÂMETROS DO ARTIGO (Valores MCMC de Martioli+2021 para AU Mic c)
# ============================================================
T0_c     = 1342.2243    # Tempo de conjunção central inicial em TBJD
T0_c_err = 0.0004       # Incerteza do T0 vinda da literatura
P_c      = 18.8588      # Período orbital em dias (AU Mic c está mais afastado que o b)
P_c_err  = 0.00012      # Incerteza do Período vinda da literatura

# Parâmetros geométricos e físicos de AU Mic c (Tabela 2)
RP_RSTAR_c = 0.0444     # Razão de raios Rp/R* (planeta c é menor que o b)
A_RSTAR_c  = 31.3       # Semieixo maior normalizado a/R* (órbita mais larga)
INC_c      = 89.04      # Inclinação orbital em graus
ECC_c      = 0.0        # Excentricidade assumida circular
W_c        = 90.0       # Argumento do periastro
LDC_c      = [0.13, 0.58] # Coeficientes de escurecimento de limbo da estrela AU Mic

# ============================================================
# 2. FUNÇÃO MODELO PARA O CURVE_FIT (Apenas o Centro T0 é Livre)
# ============================================================
def modelo_ajuste_t0_c(t_janela, t0_livre):
    """
    Função auxiliar que o curve_fit usará para testar valores de t0.
    Mantém os outros parâmetros físicos do planeta c fixados conforme o artigo.
    """
    params_c = batman.TransitParams()
    params_c.t0 = t0_livre               # Este é o único parâmetro que o algoritmo vai mudar
    params_c.per = P_c
    params_c.rp = RP_RSTAR_c
    params_c.a = A_RSTAR_c
    params_c.inc = INC_c
    params_c.ecc = ECC_c
    params_c.w = W_c
    params_c.u = LDC_c
    params_c.limb_dark = "quadratic"
    
    m_c = batman.TransitModel(params_c, t_janela)
    return m_c.light_curve(params_c)

# ============================================================
# 3. FUNÇÃO: CALCULAR CENTROS TEÓRICOS E FILTRAR TRÂNSITOS COM DADOS
# ============================================================
def mapear_transitos_validos(T0, T0_err, P, P_err, t_array, janela_dias):
    t_min, t_max = t_array.min(), t_array.max()
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    
    eventos_validos = []
    
    for n in range(n_min, n_max + 1):
        tc = T0 + n * P
        # Fórmula de propagação de erro teórica (acumula com as épocas n)
        tc_err = np.sqrt(T0_err**2 + (n * P_err)**2)
        
        # Verifica se existem pontos de dados nesta janela específica
        janela_indices = (t_array >= tc - janela_dias) & (t_array <= tc + janela_dias)
        pontos_na_janela = np.sum(janela_indices)
        
        # Só adiciona o trânsito se houver dados reais nele (> 5 pontos)
        if pontos_na_janela > 5:
            eventos_validos.append({'epoch': n, 'tc': tc, 'tc_err': tc_err})
        else:
            print(f"⚠️ Evento omitido (sem dados na lacuna): Época n={n} ao redor de BTJD={tc:.2f}")
            
    return eventos_validos

# Mapeia onde o TESS coletou dados para o planeta c usando uma janela de 0.25 dias
transitos_validos_c = mapear_transitos_validos(T0_c, T0_c_err, P_c, P_c_err, t, 0.25)

# ============================================================
# 4. AJUSTE NÃO-LINEAR E PLOTAGEM DOS GRÁFICOS INDIVIDUAIS
# ============================================================
%matplotlib qt

n_plots = len(transitos_validos_c)

if n_plots == 0:
    print("Nenhum trânsito com dados foi encontrado para AU Mic c no intervalo.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5), sharey=True)
    if n_plots == 1: axes = [axes] # Garante comportamento de lista se for apenas 1 plot

    for i, ev in enumerate(transitos_validos_c):
        ax = axes[i]
        tc_teorico = ev['tc']
        
        # Recorta estritamente os dados da janela visual atual (~6 horas para cada lado)
        janela = (t >= tc_teorico - 0.25) & (t <= tc_teorico + 0.25)
        t_f = t[janela]
        fluxo_f = residual_manchas[janela]
        
        # ─── MINIMIZAÇÃO DE MÍNIMOS QUADRADOS (AJUSTE DO CENTRO) ───
        try:
            # curve_fit ajustará o t0_livre baseado nos resíduos fotométricos reais
            # p0: chute inicial baseado na predição teórica recalculada
            # bounds: limita o algoritmo a procurar num raio seguro de ~2 horas (0.08 dias)
            popt, pcov = curve_fit(
                modelo_ajuste_t0_c, 
                t_f, 
                fluxo_f, 
                p0=[tc_teorico], 
                bounds=(tc_teorico - 0.08, tc_teorico + 0.08)
            )
            
            tc_medido = popt[0]
            # A variância do parâmetro é extraída da diagonal da matriz de covariância (pcov)
            tc_err_medido = np.sqrt(pcov[0,0])
            
            print(f"AU Mic c - Época {ev['epoch']}:")
            print(f"  T_c Teórico (Predição): {tc_teorico:.5f} d")
            print(f"  T_c Medido  (Ajustado): {tc_medido:.5f} ± {tc_err_medido:.5f} d\n")
            
        except Exception as e:
            print(f"⚠️ Não foi possível ajustar o trânsito da Época {ev['epoch']}: {e}")
            # Se o ajuste falhar por falta de convergência, mantém o teórico original como fallback
            tc_medido = tc_teorico
            tc_err_medido = ev['tc_err']
        
        # Gera a curva teórica otimizada com base no centro REAL que foi medido
        fluxo_modelo_ajustado = modelo_ajuste_t0_c(t_f, tc_medido)
        
        # Plota os dados e o novo modelo ajustado
        ax.plot(t_f, fluxo_f, 'k.', ms=3, alpha=0.4, label='Resíduos')
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5, label='Modelo c Ajustado')
        
        # Linhas de referência geométrica baseadas no centro real medido
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
        
        # ─── BARRA DE ERRO COM BASE NOS DADOS REAIS ───
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', elinewidth=2, capsize=3, 
                    ms=4, label='Incerteza $T_c$ Real ($\sigma$)')
        
        ax.set_title(f"Trânsito Real (Época {ev['epoch']})\nCentro Medido: {tc_medido:.4f} d", fontsize=10, fontweight='bold')
        ax.set_xlabel('Tempo [BTJD]', fontsize=10)
        ax.grid(alpha=0.15)
        
        if i == 0:
            ax.set_ylabel('Fluxo Residual', fontsize=12)
            ax.legend(loc='lower left', fontsize=9)

    plt.suptitle('AU Mic c — Determinação de Centros de Trânsito via Ajuste Fotométrico (Valores de Martioli+2021)', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

AU Mic c - Época 135:
  T_c Teórico (Predição): 3888.16230 d
  T_c Medido  (Ajustado): 3888.24230 ± 0.00320 d

AU Mic c - Época 136:
  T_c Teórico (Predição): 3907.02110 d
  T_c Medido  (Ajustado): 3907.10110 ± 0.00336 d



In [ ]:
# ============================================================
# 4. AJUSTE NÃO-LINEAR E PLOTAGEM DOS GRÁFICOS (APENAS TRÂNSITOS)
# ============================================================
%matplotlib qt

n_plots = len(transitos_validos_c)

if n_plots == 0:
    print("Nenhum trânsito com dados foi encontrado para AU Mic c no intervalo.")
else:
    # Layout: n_plots linhas (trânsitos empilhados verticalmente) e apenas 1 coluna
    fig, axes = plt.subplots(n_plots, 1, figsize=(6.5, 4 * n_plots), squeeze=False)

    for i, ev in enumerate(transitos_validos_c):
        ax = axes[i, 0]   # Seleciona o eixo da linha atual
        tc_teorico = ev['tc']
        
        # Recorta os dados da janela atual
        janela = (t >= tc_teorico - 0.25) & (t <= tc_teorico + 0.25)
        t_f = t[janela]
        fluxo_f = residual_manchas[janela]
        
        # ─── MINIMIZAÇÃO DE MÍNIMOS QUADRADOS (AJUSTE DO CENTRO) ───
        try:
            popt, pcov = curve_fit(
                modelo_ajuste_t0_c, 
                t_f, 
                fluxo_f, 
                p0=[tc_teorico], 
                bounds=(tc_teorico - 0.08, tc_teorico + 0.08)
            )
            
            tc_medido = popt[0]
            tc_err_medido = np.sqrt(pcov[0,0])
            
            print(f"AU Mic c - Época {ev['epoch']}:")
            print(f"  T_c Teórico (Predição): {tc_teorico:.5f} d")
            print(f"  T_c Medido  (Ajustado): {tc_medido:.5f} ± {tc_err_medido:.5f} d\n")
            
        except Exception as e:
            print(f"⚠️ Não foi possível ajustar o trânsito da Época {ev['epoch']}: {e}")
            tc_medido = tc_teorico
            tc_err_medido = ev['tc_err']
        
        # Gera a curva teórica otimizada com base no centro real medido
        fluxo_modelo_ajustado = modelo_ajuste_t0_c(t_f, tc_medido)
        
        # ─── DELIMITAÇÃO AUTOMÁTICA DO INÍCIO E FIM DO TRÂNSITO ───
        em_transito = fluxo_modelo_ajustado < 0.99999
        if np.any(em_transito):
            t_inicio = t_f[em_transito].min()
            t_fim = t_f[em_transito].max()
            
            # Linhas verticais azuis pontilhadas marcando o ingresso e egresso
            ax.axvline(t_inicio, color='royalblue', ls=':', lw=1.5, label='Início/Fim Trânsito' if i==0 else "")
            ax.axvline(t_fim, color='royalblue', ls=':', lw=1.5)
        
        # ─── PLOTAGEM DA CURVA DE LUZ ───
        # Aplicado o padrão 'k.-' exigido, usando uma linha fina e transparência para destacar o modelo
        ax.plot(t_f, fluxo_f, 'k.-', ms=4, lw=0.6, alpha=0.4, label='Dados (Sem Manchas)')
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5, label='Modelo c Ajustado')
        
        # Linhas de referência geométrica (Centro medido e Linha de base 1.0)
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
        
        # Barra de erro do Tc ajustado no topo do trânsito
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', elinewidth=2, capsize=3, 
                    ms=4, label='Incerteza $T_c$ Real ($\sigma$)')
        
        # Customização de rótulos e eixos
        ax.set_title(f"AU Mic c — Trânsito Real (Época {ev['epoch']}) — Centro Medido: {tc_medido:.4f} d", fontsize=10, fontweight='bold')
        ax.set_xlabel('Tempo [BTJD]', fontsize=9)
        ax.set_ylabel('Fluxo Residual', fontsize=10)
        ax.grid(alpha=0.15)
        
        # Exibe a legenda apenas no primeiro gráfico para poupar espaço
        if i == 0: 
            ax.legend(loc='lower left', fontsize=8)

    plt.suptitle('AU Mic c — Determinação de Centros de Trânsito via Ajuste Fotométrico (Valores de Martioli+2021)', fontsize=11, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

AU Mic c - Época 135:
  T_c Teórico (Predição): 3888.16230 d
  T_c Medido  (Ajustado): 3888.24230 ± 0.00320 d

AU Mic c - Época 136:
  T_c Teórico (Predição): 3907.02110 d
  T_c Medido  (Ajustado): 3907.10110 ± 0.00336 d



: 